### DEMO

In [1]:
!pip install --upgrade --quiet accelerate bitsandbytes transformers
!pip -q install fastapi uvicorn pyngrok python-multipart pillow

In [2]:
from transformers import BitsAndBytesConfig
import torch

model_variant = "medgemma-1.5-4b-it"
model_id = f"google/{model_variant}"

model_kwargs = dict(
    device_map="auto",
    dtype=torch.bfloat16,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True),
)

In [3]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

In [4]:
from transformers import AutoModelForImageTextToText, AutoProcessor

model = AutoModelForImageTextToText.from_pretrained(model_id, **model_kwargs)
processor = AutoProcessor.from_pretrained(model_id)

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [5]:
import io, json, re, uuid, torch
from fastapi import FastAPI, File, UploadFile, Form, Body, HTTPException
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Any
from PIL import Image

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── In-memory session store ──
sessions: dict[str, bytes] = {}

# ═══════════════════════════════════════════════════════════════
# MEDGEMMA INFERENCE HELPERS
# ═══════════════════════════════════════════════════════════════

def run_medgemma_image(
    pil_img: Image.Image,
    prompt_text: str,
    role: str,
    max_tokens: int = 512
) -> str:

    messages = [
        {"role": "system", "content": [{"type": "text", "text": role}]},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": pil_img},
                {"type": "text", "text": prompt_text},
            ],
        },
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )

    output = output[0][input_len:]

    return processor.decode(output, skip_special_tokens=True)


# ═══════════════════════════════════════════════════════════════
# MODELS
# ═══════════════════════════════════════════════════════════════

class AssessmentSubmit(BaseModel):
    sessionId: str
    answers: list[dict[str, Any]]


# ═══════════════════════════════════════════════════════════════
# HEALTH
# ═══════════════════════════════════════════════════════════════

@app.get("/health")
def health():
    return {"status": "ok"}


# ═══════════════════════════════════════════════════════════════
# START ASSESSMENT (UPLOAD + VALIDATION + QUESTIONS)
# ═══════════════════════════════════════════════════════════════

@app.post("/api/assessment/start")
async def assessment_start(image: UploadFile = File(...)):

    img_bytes = await image.read()
    session_id = str(uuid.uuid4())
    sessions[session_id] = img_bytes

    pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")

    # -------- VALIDACIÓN SUAVE --------
    validation_prompt = """
Is this a clear image of the inner lower eyelid (palpebral conjunctiva)?
Respond ONLY in JSON:
{"valid": true/false, "reason": "short Spanish explanation"}
"""

    raw_validation = run_medgemma_image(
        pil_img,
        validation_prompt,
        "You are a medical image validator.",
        max_tokens=200
    )

    json_match = re.search(r'\{[\s\S]*?\}', raw_validation)

    valid = True
    reason = "Imagen aceptada."

    if json_match:
        try:
            parsed = json.loads(json_match.group())
            valid = parsed.get("valid", True)
            reason = parsed.get("reason", reason)
        except:
            pass

    # -------- GENERACIÓN DE PREGUNTAS --------
    questions_prompt = """
Based on this conjunctiva image,
generate 5 short yes/no clinical questions relevant to pediatric anemia.
Return as numbered list.
"""

    raw_questions = run_medgemma_image(
        pil_img,
        questions_prompt,
        "You are a pediatric anemia specialist.",
        max_tokens=400
    )

    questions = []
    for line in raw_questions.split("\n"):
        line = line.strip()
        if len(line) > 15:
            questions.append({
                "id": f"q_{uuid.uuid4().hex[:6]}",
                "type": "yes_no",
                "text": line
            })
        if len(questions) >= 5:
            break

    return JSONResponse({
        "sessionId": session_id,
        "imageValid": valid,
        "validationMessage": reason,
        "questions": questions
    })


# ═══════════════════════════════════════════════════════════════
# SUBMIT FINAL EVALUATION
# ═══════════════════════════════════════════════════════════════

@app.post("/api/assessment/submit")
def assessment_submit(body: AssessmentSubmit):

    img_bytes = sessions.get(body.sessionId)

    if not img_bytes:
        raise HTTPException(status_code=404, detail="Session not found")

    pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")

    answers_block = "\n".join(
        f"- {a.get('questionId')}: {a.get('value')}"
        for a in body.answers
    )

    final_prompt = f"""
Analyze conjunctiva image and these answers:

{answers_block}

Return ONLY valid JSON:
{{
  "riskLevel": "low | medium | high",
  "explanation": "Spanish explanation",
  "recommendations": ["rec1", "rec2"]
}}
"""

    raw = run_medgemma_image(
        pil_img,
        final_prompt,
        "You are a pediatric anemia specialist. Return only JSON.",
        max_tokens=800
    )

    json_match = re.search(r'\{[\s\S]*?\}', raw)

    if not json_match:
        return JSONResponse({
            "riskLevel": "medium",
            "explanation": "No se pudo interpretar completamente el análisis.",
            "recommendations": ["Consulte con un profesional de salud."]
        })

    try:
        data = json.loads(json_match.group())
        return data
    except:
        return JSONResponse({
            "riskLevel": "medium",
            "explanation": "Error interpretando la respuesta del modelo.",
            "recommendations": ["Consulte con un profesional de salud."]
        })







print("✅ FastAPI app ready (stable version)")

✅ FastAPI app ready (stable version)


In [6]:
print("run_medgemma_image" in globals())

True


In [7]:
import threading
import uvicorn

def run():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info",
        loop="auto",        # más estable en notebooks
        http="h11",
        access_log=False
    )

thread = threading.Thread(target=run, daemon=True)
thread.start()

print("✅ Uvicorn thread started")

✅ Uvicorn thread started


In [8]:
from google.colab import userdata
from pyngrok import ngrok

ngrok_token = userdata.get("NGROK_TOKEN")

if not ngrok_token:
    raise ValueError("❌ NGROK_TOKEN no está configurado en Colab secrets.")

ngrok.set_auth_token(ngrok_token)

print("✅ Ngrok token configurado correctamente")

INFO:     Started server process [61719]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ Ngrok token configurado correctamente


In [9]:
from pyngrok import ngrok

# Cerrar túneles previos de forma segura
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

public_url = ngrok.connect(8000)

print("✅ Public URL:", public_url.public_url)

✅ Public URL: https://sandi-sphygmomanometric-nonvituperatively.ngrok-free.dev


In [10]:
!curl --max-time 5 -s -o /dev/null -w "%{http_code}\n" http://127.0.0.1:8000/health

200


In [11]:
url = public_url.public_url + "/predict"
print(url)

https://sandi-sphygmomanometric-nonvituperatively.ngrok-free.dev/predict


# PRUEBAS

In [12]:
import requests
import time

BASE = public_url.public_url

time.sleep(1)  # pequeño margen por si ngrok aún está iniciando

resp = requests.get(f"{BASE}/health", timeout=5)
print(resp.status_code, resp.json())

200 {'status': 'ok'}


In [13]:
print([route.path for route in app.routes])

['/openapi.json', '/docs', '/docs/oauth2-redirect', '/redoc', '/health', '/api/assessment/start', '/api/assessment/submit']


In [14]:
import requests
import json
from PIL import Image

BASE = public_url.public_url

# --- Crear imagen de prueba ---
img = Image.new("RGB", (400, 300), color=(220, 180, 160))
img.save("test_eye.jpg", "JPEG")
print("Imagen test_eye.jpg creada\n")

# === STEP 1: Upload + Questions ===
with open("test_eye.jpg", "rb") as f:
    resp = requests.post(
        f"{BASE}/api/assessment/start",
        files={"image": ("test_eye.jpg", f, "image/jpeg")}
    )

print("START:", resp.status_code)
data = resp.json()
print(json.dumps(data, indent=2, ensure_ascii=False))

session_id = data["sessionId"]
questions = data["questions"]

# === STEP 2: Submit Answers ===
test_answers = []

for q in questions:
    test_answers.append({
        "questionId": q["id"],
        "value": True
    })

print("\nSUBMIT: esperando respuesta final...\n")

resp = requests.post(
    f"{BASE}/api/assessment/submit",
    json={
        "sessionId": session_id,
        "answers": test_answers
    }
)

print("FINAL:", resp.status_code)
print(json.dumps(resp.json(), indent=2, ensure_ascii=False))

Imagen test_eye.jpg creada



Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


START: 200
{
  "sessionId": "38c9d572-2ebc-456f-9323-fd88a5747059",
  "imageValid": false,
  "validationMessage": "The image does not show the inner lower eyelid (palpebral conjunctiva). It appears to be a solid pink color, possibly a skin tone or a texture sample, not an image of a biological structure like the inner lower eyelid.",
  "questions": [
    {
      "id": "q_df9bbc",
      "type": "yes_no",
      "text": "Here are 5 relevant yes/no clinical questions based on the provided conjunctiva image, relevant to pediatric anemia:"
    },
    {
      "id": "q_a6c703",
      "type": "yes_no",
      "text": "1.  Is the child experiencing symptoms like fatigue or pallor?"
    },
    {
      "id": "q_092608",
      "type": "yes_no",
      "text": "2.  Is the child's growth pattern normal for their age?"
    },
    {
      "id": "q_05e032",
      "type": "yes_no",
      "text": "3.  Has the child had recent blood loss (e.g., from nosebleeds, vomiting)?"
    },
    {
      "id": "q_d050b9"

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


FINAL: 200
{
  "riskLevel": "low",
  "explanation": "The provided image shows a normal conjunctiva. There are no signs of anemia or other abnormalities.",
  "recommendations": [
    "continue monitoring",
    "further investigation not required"
  ]
}


In [15]:
for i, q in enumerate(questions, 1):
    print(f"  {i}. [{q.get('type')}] {q.get('text', q.get('id', ''))}")

  1. [yes_no] Here are 5 relevant yes/no clinical questions based on the provided conjunctiva image, relevant to pediatric anemia:
  2. [yes_no] 1.  Is the child experiencing symptoms like fatigue or pallor?
  3. [yes_no] 2.  Is the child's growth pattern normal for their age?
  4. [yes_no] 3.  Has the child had recent blood loss (e.g., from nosebleeds, vomiting)?
  5. [yes_no] 4.  Is the child consuming adequate iron-rich foods?
